# Promoter Reporter Analysis

This notebook collects fluorescence intensity data from promoter reporter experiments comparing control and pSEVA conditions.

## Import Libraries

In [1]:
import pathlib
import pandas as pd

## Define Data Loading Function

Function to load fluorescence intensity data from each position folder and aggregate statistics

In [2]:
def load_condition(root_path, condition_name):
    rows = []

    rep_folders = sorted(
        [p for p in root_path.glob("rep*") if p.is_dir()]
    )

    for rep in rep_folders:
        pos_folders = sorted(
                [p for p in rep.glob("pos*") if p.is_dir()]
            )
        
        for pos in pos_folders:
            csv_path = pos / "single_cell_props.csv"
            if not csv_path.exists():
                continue

            df_pos = pd.read_csv(csv_path)

            rows.append({
                "condition": condition_name,
                "pos": pos.name,
                "raw_gfp_intensity": df_pos["intensity_raw_gfp"].mean(),
                "mcherry_raw_intensity": df_pos["intensity_raw_mcherry"].mean(),
                "n_cells": len(df_pos)
            })

    return pd.DataFrame(rows)


## Load Data

Load fluorescence intensity data for control and pSEVA conditions from the Image_Data directory

In [3]:

# set path to BioImageArchive data directory:
data_archive_path = pathlib.Path('/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/')

# input and output paths relative to BioImageArchive location
image_data_root = data_archive_path / 'PAReporter'

paths = {
    "control": image_data_root / "S2_wt",
    "pSEVA": image_data_root / "S2_wt+pSEVA",
}

# Load data for each condition
df_control = load_condition(paths["control"], "control")
df_pseva = load_condition(paths["pSEVA"], "pSEVA")

# Combine into single dataframe
df = pd.concat([df_control, df_pseva], ignore_index=True)

print(f"Loaded {len(df_control)} control positions and {len(df_pseva)} pSEVA positions")
print(f"Total cells: control={df_control['n_cells'].sum()}, pSEVA={df_pseva['n_cells'].sum()}")

df.to_csv("reporter_intensity_summary.csv", index=False)

Loaded 9 control positions and 10 pSEVA positions
Total cells: control=4215, pSEVA=5733
